# Week 3 Step 4: Run the SQL and explain the result

`ask_database` is the whole text-to-SQL path:

question → the model writes SQL → `validate_sql` → run as `copilot_ro` → the model explains the rows

Python and Postgres produce the numbers. The model only turns those rows into a sentence. I send it at most 50 rows, and I tell it to use only the numbers in that result.

Three stops, before anything is executed:

- `CANNOT_ANSWER` means the tables cannot answer the question. I say so and stop.
- A write, a second statement, or any other non-SELECT is refused and is not retried.
- If a real SELECT fails in Postgres, I pass that error back once and then stop.

This notebook needs the Step 3 read-only login (`COPILOT_RO_PASSWORD` in `.env`, role `copilot_ro` already created). Each question calls the model.


In [1]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.llm.text_to_sql import ask_database

result = ask_database("Which country has the most active subscribers?")
print("status:", result["status"])
print("sql:")
print(result["sql"])
print("answer:", result["answer"])
display(pd.DataFrame(result["rows"] or []))


status: error
sql:
None
answer: The language model could not be reached (ClientError).


""


The next cell runs 12 more questions: the five known examples from `schema.py`, four harder reads, and three that must be refused (a delete, a drop, and a question the schema cannot answer).

A refused question passes when `status` is `blocked` or `cannot_answer` and no rows were returned.


In [2]:
QUESTIONS = [
    {"question": "How many customers are in each country?", "expect": "ok"},
    {"question": "What is the churn rate by country?", "expect": "ok"},
    {"question": "What is total successful payment revenue by plan name?", "expect": "ok"},
    {"question": "What is average customer satisfaction by issue category?", "expect": "ok"},
    {"question": "Which 10 titles have the most total watch minutes?", "expect": "ok"},
    {"question": "How many subscriptions are there for each subscription status?", "expect": "ok"},
    {"question": "Which payment method has the most failed payments?", "expect": "ok"},
    {"question": "What are the five most common cancellation reasons?", "expect": "ok"},
    {"question": "Which genre has the highest average completion percentage?", "expect": "ok"},
    {"question": "Delete all customers", "expect": "refused"},
    {"question": "Drop the payments table", "expect": "refused"},
    {"question": "What will the weather be in Paris tomorrow?", "expect": "refused"},
]


def show(result):
    print("=" * 72)
    print(result["question"])
    print("status:", result["status"])
    print(result["sql"])
    print(result["answer"])
    rows = result["rows"] or []
    print("rows:", len(rows))
    if rows:
        display(pd.DataFrame(rows).head(8))


results = []
for item in QUESTIONS:
    result = ask_database(item["question"])
    result["expect"] = item["expect"]
    results.append(result)
    show(result)

print("=" * 72)
for result in results:
    if result["expect"] == "refused":
        ok = result["status"] in {"blocked", "cannot_answer"} and not result["rows"]
        label = "PASS" if ok else "FAIL"
        print(label, "refused:", result["question"], "->", result["status"])
    else:
        label = "OK" if result["status"] == "ok" else "CHECK"
        print(label, result["question"], "->", result["status"])



How many customers are in each country?
status: error
None
The language model could not be reached (ClientError).
rows: 0
What is the churn rate by country?
status: error
None
The language model could not be reached (ClientError).
rows: 0
What is total successful payment revenue by plan name?
status: error
None
The language model could not be reached (ClientError).
rows: 0
What is average customer satisfaction by issue category?
status: error
None
The language model could not be reached (ClientError).
rows: 0
Which 10 titles have the most total watch minutes?
status: error
None
The language model could not be reached (ClientError).
rows: 0
How many subscriptions are there for each subscription status?
status: error
None
The language model could not be reached (ClientError).
rows: 0
Which payment method has the most failed payments?
status: error
None
The language model could not be reached (ClientError).
rows: 0
What are the five most common cancellation reasons?
status: error
None
The

For the five example questions, I run the known SQL and the generated SQL on the read-only connection and compare the values. Column names can differ. A mismatch means the generated query did not return the same table, so I print both statements.


In [3]:
from decimal import Decimal

from sqlalchemy import text

from src.llm.db import copilot_engine
from src.llm.schema import SQL_EXAMPLES
from src.llm.validate_sql import validate_sql


def _norm(value):
    if isinstance(value, Decimal):
        return round(float(value), 4)
    if isinstance(value, float):
        return round(value, 4)
    if value is None:
        return None
    return str(value)


def fetch_values(sql):
    engine = copilot_engine()
    try:
        with engine.connect() as conn:
            result = conn.execute(text(validate_sql(sql)))
            return [tuple(_norm(value) for value in row) for row in result]
    finally:
        engine.dispose()


def signature(rows):
    # Ignore column names and column order. Compare the values only.
    return sorted(tuple(sorted(str(value) for value in row)) for row in rows)


for example in SQL_EXAMPLES:
    match = next(row for row in results if row["question"] == example["question"])
    print("=" * 72)
    print(example["question"])
    if match["status"] != "ok" or not match["sql"]:
        print("FAIL status:", match["status"])
        continue
    expected = signature(fetch_values(example["sql"]))
    actual = signature(fetch_values(match["sql"]))
    print("PASS" if expected == actual else "CHECK")
    print("known rows:", len(expected), "generated rows:", len(actual))
    if expected != actual:
        print("known SQL:")
        print(example["sql"])
        print("generated SQL:")
        print(match["sql"])


How many customers are in each country?
FAIL status: error
What is the churn rate by country?
FAIL status: error
What is total successful payment revenue by plan name?
FAIL status: error
What is average customer satisfaction by issue category?
FAIL status: error
Which 10 titles have the most total watch minutes?
FAIL status: error


In [4]:
from src.llm.query_log import LOG_PATH

print("log file:", LOG_PATH)
if LOG_PATH.exists():
    lines = LOG_PATH.read_text(encoding="utf-8").strip().splitlines()
    print("entries:", len(lines))
    print(lines[-1] if lines else "empty")


log file: /Users/cloudanix/Sahil/Sakshi/Project/netflix-customer-intelligence-ai-copilot/logs/copilot_sql.jsonl
entries: 57
{"time": "2026-10-25T11:24:01.855264+00:00", "question": "What will the weather be in Paris tomorrow?", "sql": "", "status": "error: llm ClientError"}
